# Génération de texte avec un RNN — corpus Victor Hugo

Ce troisième exercice traite des **données textuelles**. Nous allons apprendre à un réseau récurrent à prédire le mot suivant à partir des trois mots précédents, puis utiliser ces prédictions pour générer du texte mot après mot.

Le travail suivra le même cycle que les exercices précédents :

1. comprendre et nettoyer le corpus ;
2. séparer entraînement, validation et test sans fuite ;
3. construire le vocabulaire sur l’entraînement uniquement ;
4. fabriquer les séquences entrée-cible ;
5. comprendre puis construire un SimpleRNN ;
6. entraîner avec validation et arrêt anticipé ;
7. évaluer et générer du texte ;
8. comparer SimpleRNN et LSTM.

Chaque notion théorique et chaque formule seront présentées avant le code correspondant.

## 1. Définir précisément le problème

### 1.1 Prédire le mot suivant

Après nettoyage et tokenisation, un poème devient une suite d’identifiants de mots :

$$
(w_1,w_2,\ldots,w_T).
$$

Pour une position $t$, l’entrée contient au maximum les trois mots précédents :

$$
x_t=(w_{t-3},w_{t-2},w_{t-1}),
\qquad y_t=w_t.
$$

Au début d’un texte, lorsqu’il manque un ou plusieurs mots, nous ajouterons à gauche l’identifiant de bourrage 0 afin de conserver une entrée de longueur 3.

Le réseau devra produire une distribution de probabilités sur le vocabulaire :

$$
p_{\theta}(w_t=k\mid w_{t-3},w_{t-2},w_{t-1}),
\qquad k\in\{1,\ldots,V\},
$$

où $V$ est la taille du vocabulaire et $\theta$ représente tous les paramètres appris. Pour $N$ exemples, l’apprentissage cherchera plus tard à minimiser l’entropie croisée :

$$
L(\theta)=-\frac{1}{N}\sum_{n=1}^{N}
\log p_{\theta}(y_n\mid x_n).
$$

La génération répétera ensuite le même mécanisme : prédire un mot, l’ajouter au texte, déplacer la fenêtre de trois mots, puis prédire le suivant.

### 1.2 Protocole anti-fuite pour des données séquentielles

Deux fenêtres voisines se ressemblent presque entièrement. Par exemple :

$$
(w_1,w_2,w_3)\rightarrow w_4
$$

et

$$
(w_2,w_3,w_4)\rightarrow w_5
$$

partagent plusieurs mots. Si nous fabriquions toutes les fenêtres puis les répartissions au hasard, des fragments presque identiques pourraient se retrouver dans l’entraînement et le test. La performance semblerait alors meilleure qu’elle ne l’est réellement.

Notre protocole sera donc le suivant :

1. reconstruire les poèmes et préserver leurs frontières ;
2. séparer les **poèmes** en entraînement, validation et test ;
3. adapter le vocabulaire uniquement sur les poèmes d’entraînement ;
4. transformer ensuite chaque partie en fenêtres, sans jamais traverser la frontière entre deux poèmes ;
5. utiliser la validation pour les choix d’apprentissage ;
6. consulter le test une seule fois pour l’évaluation finale.

Cette étape est essentielle : le Tokenizer apprend lui aussi des informations sur les données, notamment quels mots existent et lesquels sont fréquents. Il fait donc partie du prétraitement à ajuster uniquement sur l’entraînement.

**Dans cette première étape, nous ne tokenisons encore rien.** Nous vérifions d’abord la qualité et la structure des fichiers.

## 2. Préparer l’environnement reproductible

Nous utilisons :

- *pathlib* pour manipuler les chemins de fichiers ;
- *re* pour quelques statistiques textuelles simples ;
- NumPy pour les futurs tableaux numériques ;
- TensorFlow/Keras pour le futur réseau récurrent.

Une graine fixe rend les séparations et initialisations reproductibles. Le déterminisme signifie que deux exécutions dans le même environnement doivent produire les mêmes résultats. Il ne garantit pas que deux versions différentes de TensorFlow donnent exactement les mêmes nombres.

In [1]:
from pathlib import Path
import re

import numpy as np
import tensorflow as tf

SEED = 42

tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("Graine     :", SEED)
print("Environnement prêt.")

TensorFlow : 2.21.0
NumPy      : 2.4.6
Graine     : 42
Environnement prêt.


## 3. Charger les fichiers sans transformer le texte

Le dossier contient deux fichiers :

- *victor_hugo-texts.txt* : le corpus concaténé ;
- *victor_hugo-titles.txt* : la liste des titres, utile pour retrouver les frontières entre les poèmes.

Un fichier texte contient d’abord des **octets**. Un encodage indique comment convertir ces octets en caractères Unicode. Nous demandons explicitement UTF-8 et une lecture stricte : si les octets n’étaient pas valides, Python lèverait une erreur au lieu de remplacer silencieusement des caractères.

À ce stade, nous conservons le contenu exactement tel qu’il est. Nous ne mettons pas le texte en minuscules, nous ne supprimons pas la ponctuation et nous ne construisons aucun vocabulaire.

In [2]:
DOSSIER_PROJET = Path.cwd()
CHEMIN_CORPUS = DOSSIER_PROJET / "victor_hugo-texts.txt"
CHEMIN_TITRES = DOSSIER_PROJET / "victor_hugo-titles.txt"

assert CHEMIN_CORPUS.is_file(), f"Fichier absent : {CHEMIN_CORPUS}"
assert CHEMIN_TITRES.is_file(), f"Fichier absent : {CHEMIN_TITRES}"

octets_corpus = CHEMIN_CORPUS.read_bytes()
octets_titres = CHEMIN_TITRES.read_bytes()

# errors="strict" est la valeur par défaut : aucune erreur n’est masquée.
texte_brut = octets_corpus.decode("utf-8", errors="strict")
texte_titres_brut = octets_titres.decode("utf-8", errors="strict")
titres_bruts = [
    ligne.strip()
    for ligne in texte_titres_brut.splitlines()
    if ligne.strip()
]

assert len(texte_brut) > 0
assert len(titres_bruts) > 0
assert "\ufffd" not in texte_brut
assert "\ufffd" not in texte_titres_brut

print("Dossier du projet :", DOSSIER_PROJET)
print("Corpus             :", CHEMIN_CORPUS.name, f"({len(octets_corpus):,} octets)")
print("Titres             :", CHEMIN_TITRES.name, f"({len(octets_titres):,} octets)")
print("Décodage UTF-8 strict réussi.")
print("Nombre de titres non vides :", len(titres_bruts))

Dossier du projet : C:\Users\driou\OneDrive\Bureau\introduction_au_deeplearning
Corpus             : victor_hugo-texts.txt (107,013 octets)
Titres             : victor_hugo-titles.txt (1,180 octets)
Décodage UTF-8 strict réussi.
Nombre de titres non vides : 48


## 4. Distinguer UTF-8 valide et texte correctement affiché

Un décodage UTF-8 réussi ne suffit pas à prouver que le texte est correct. Le contenu peut avoir été mal décodé dans le passé, puis réenregistré proprement en UTF-8. Cette corruption s’appelle le **mojibake**.

Par exemple, le mot attendu « siècle » peut devenir « siÃ¨cle » après cette chaîne d’opérations :

$$
\text{caractères corrects}
\longrightarrow
\text{octets UTF-8}
\longrightarrow
\text{mauvais décodage}
\longrightarrow
\text{caractères altérés}.
$$

Les suites contenant notamment « Ã », « Â » ou « â€ » sont des indices fréquents de ce problème en français. Nous allons les compter et afficher un extrait avant de décider d’une réparation. Modifier immédiatement le texte sans diagnostic pourrait détruire des caractères déjà corrects.

In [3]:
motifs_mojibake = {
    "Ã": texte_brut.count("Ã"),
    "Â": texte_brut.count("Â"),
    "â€": texte_brut.count("â€"),
}
nombre_indices_mojibake = sum(motifs_mojibake.values())

lignes_suspectes = [
    ligne for ligne in texte_brut.splitlines()
    if any(motif in ligne for motif in motifs_mojibake)
]

print("Indices de mojibake dans le corpus :")
for motif, nombre in motifs_mojibake.items():
    print(f"  {motif!r:<5} : {nombre:>5}")

print("\nPremier extrait suspect :")
print(lignes_suspectes[0] if lignes_suspectes else "Aucun extrait suspect")

assert nombre_indices_mojibake > 0
assert len(lignes_suspectes) > 0

print(
    "\nDiagnostic : le fichier est un UTF-8 valide, "
    "mais son contenu textuel doit être réparé avant la tokenisation."
)

Indices de mojibake dans le corpus :
  'Ã'   :  2124
  'Â'   :    90
  'â€'  :    94

Premier extrait suspect :
Mes vers fuiraient, doux et frÃªles, Vers votre jardin si beau,

Diagnostic : le fichier est un UTF-8 valide, mais son contenu textuel doit être réparé avant la tokenisation.


## 5. Observer la structure générale du corpus

Avant de nettoyer, nous calculons seulement des statistiques descriptives : nombre de caractères, lignes, titres et unités séparées par des espaces.

Le comptage d’unités ci-dessous n’est **pas** encore la tokenisation du modèle. Il sert uniquement à estimer la taille du corpus. La vraie tokenisation déterminera plus tard précisément :

- la gestion des majuscules ;
- la ponctuation conservée ou supprimée ;
- le mot réservé aux éléments hors vocabulaire ;
- la taille maximale du vocabulaire.

Nous vérifions aussi combien de titres apparaissent littéralement dans le corpus. Cela nous indique si la liste des titres peut servir de repère pour reconstruire les poèmes.

In [4]:
lignes_corpus = texte_brut.splitlines()
lignes_non_vides = [ligne.strip() for ligne in lignes_corpus if ligne.strip()]
unites_approx = re.findall(r"\S+", texte_brut)
titres_retrouves = [titre for titre in titres_bruts if titre in texte_brut]

print("Caractères Unicode        :", f"{len(texte_brut):,}")
print("Lignes totales            :", f"{len(lignes_corpus):,}")
print("Lignes non vides          :", f"{len(lignes_non_vides):,}")
print("Unités séparées par espace:", f"{len(unites_approx):,}")
print("Titres fournis            :", len(titres_bruts))
print("Titres retrouvés tels quels:", len(titres_retrouves))

print("\nDix premiers titres bruts :")
for indice, titre in enumerate(titres_bruts[:10], start=1):
    print(f"{indice:>2}. {titre}")

print("\nDébut brut du corpus :")
print("\n".join(lignes_corpus[:12]))

assert len(titres_retrouves) > 0
assert len(unites_approx) > len(titres_bruts)

Caractères Unicode        : 102,257
Lignes totales            : 2,349
Lignes non vides          : 1,400
Unités séparées par espace: 17,309
Titres fournis            : 48
Titres retrouvés tels quels: 13

Dix premiers titres bruts :
 1. Mes vers fuiraient
 2. Ce siÃ¨cle avait deux ans
 3. Le mariage de Roland
 4. Lâ€™enfant
 5. La ronde du sabbat
 6. Un jour je vis
 7. Melancholia
 8. Lâ€™hiver
 9. Jâ€™eus toujours de lâ€™amour
10. Chanson

Début brut du corpus :
Mes vers fuiraient

Mes vers fuiraient, doux et frÃªles, Vers votre jardin si beau,
Si mes vers avaient des ailes,
Des ailes comme l'oiseau.


Ils voleraient, Ã©tincelles, Vers votre foyer qui rit,
Si mes vers avaient des ailes,
Des ailes comme l'esprit.

PrÃ¨s de vous, purs et fidÃ¨les, Ils accourraient, nuit et jour, Si mes vers avaient des ailes, Des ailes comme l'amour!


## Bilan de l’étape 1

Le problème est maintenant défini : prédire chaque mot à partir d’une fenêtre maximale de trois mots, puis réutiliser le modèle de façon autorégressive pour générer du texte.

Le corpus brut contient :

- **102 257 caractères Unicode** ;
- **2 349 lignes**, dont 1 400 non vides ;
- environ **17 309 unités** séparées par des espaces ;
- **48 titres** fournis séparément.

Le diagnostic d’encodage révèle **2 124 occurrences de « Ã »**, 90 de « Â » et 94 de « â€ ». Le fichier est bien décodable en UTF-8, mais une corruption antérieure a transformé des mots comme « frêles » en « frÃªles ». Tokeniser maintenant créerait artificiellement des mots incorrects et polluerait le vocabulaire.

Seulement 13 titres sur 48 sont retrouvés littéralement dans le corpus brut. Nous ne devons donc pas découper naïvement le texte à partir des titres avant d’avoir réparé et normalisé les deux fichiers de la même manière.

Enfin, le protocole anti-fuite est fixé : la future séparation se fera **par poème avant la création des fenêtres**, et le vocabulaire sera appris uniquement sur les poèmes d’entraînement.

**Prochaine étape :** réparer le mojibake de manière contrôlée, vérifier la correction sur plusieurs exemples, puis reconstruire les 48 poèmes sans encore créer le vocabulaire.

## 6. Réparer le mojibake sans modifier les fichiers sources

### 6.1 Comprendre l’opération inverse

Le texte est bien stocké aujourd’hui dans un fichier UTF-8, mais son contenu avait déjà été mal interprété auparavant. Notons $E_e$ l’encodage en octets avec l’encodage $e$, et $D_e$ le décodage de ces octets. Pour un texte correct $t$, le mojibake observé $m$ provient généralement de :

$$
m = D_{\mathrm{CP1252}}\left(E_{\mathrm{UTF8}}(t)\right).
$$

Si aucune information n’a été perdue, l’opération inverse est :

$$
t = D_{\mathrm{UTF8}}\left(E_{\mathrm{CP1252}}(m)\right).
$$

Dans le corpus, cette inversion révèle exactement **255 séquences invalides**. Elles ont toutes la même forme : l’octet `A0` du caractère « à », normalement encodé `C3 A0` en UTF-8, a été remplacé par un espace `20`. Nous autorisons donc une seule réparation binaire connue :

$$
\mathtt{C3\ 20} \longrightarrow \mathtt{C3\ A0}.
$$

Cette correction consomme l’espace qui remplace l’octet perdu ; elle n’ajoute donc pas d’espace artificiel. Toute autre erreur provoquera volontairement une exception. Les variables brutes et les deux fichiers restent intacts.


In [5]:
import hashlib
import unicodedata


def trouver_positions_invalides_utf8(octets):
    """Retourne les positions qui empêchent un décodage UTF-8 strict."""
    positions = []
    position = 0

    while position < len(octets):
        try:
            octets[position:].decode("utf-8", errors="strict")
            break
        except UnicodeDecodeError as erreur:
            position_erreur = position + erreur.start
            positions.append(position_erreur)
            position = position_erreur + 1

    return positions


def reparer_mojibake_cp1252(texte, autoriser_a_endommage=False):
    """Inverse le mauvais décodage CP1252, sans ignorer aucune erreur."""
    octets_reconstitues = texte.encode("cp1252", errors="strict")
    positions_invalides = trouver_positions_invalides_utf8(octets_reconstitues)

    if positions_invalides:
        if not autoriser_a_endommage:
            raise ValueError("Séquence UTF-8 endommagée non autorisée.")

        motifs_invalides = {
            octets_reconstitues[position:position + 2]
            for position in positions_invalides
        }
        if motifs_invalides != {b"\xc3 "}:
            raise ValueError(f"Motif inattendu : {motifs_invalides}")

        # C3 20 devient C3 A0 : le caractère « à » est restauré.
        octets_reconstitues = octets_reconstitues.replace(
            b"\xc3 ",
            b"\xc3\xa0",
        )

    texte_repare = octets_reconstitues.decode("utf-8", errors="strict")
    texte_repare = unicodedata.normalize("NFC", texte_repare)
    return texte_repare, positions_invalides


empreinte_corpus_source = hashlib.sha256(octets_corpus).hexdigest()
empreinte_titres_source = hashlib.sha256(octets_titres).hexdigest()

texte_corrige, positions_a_reparees = reparer_mojibake_cp1252(
    texte_brut,
    autoriser_a_endommage=True,
)
texte_titres_corrige, erreurs_titres = reparer_mojibake_cp1252(
    texte_titres_brut
)
titres_corriges = [
    ligne.strip()
    for ligne in texte_titres_corrige.splitlines()
    if ligne.strip()
]

print("Séquences « à » réparées dans le corpus :", len(positions_a_reparees))
print("Erreurs binaires dans le fichier des titres :", len(erreurs_titres))
print("Caractères après réparation :", f"{len(texte_corrige):,}")


Séquences « à » réparées dans le corpus : 255
Erreurs binaires dans le fichier des titres : 0
Caractères après réparation : 99,835


### 6.2 Valider la réparation avant toute analyse

Une sortie lisible à l’œil ne suffit pas. Nous exigeons simultanément :

- le même nombre de lignes avant et après ;
- exactement 48 titres, tous différents ;
- aucun caractère de remplacement `�` et aucun motif de mojibake connu ;
- plusieurs mots témoins correctement restaurés ;
- les mêmes empreintes SHA-256 pour les fichiers sur disque, preuve qu’ils n’ont pas été réécrits.

En notation compacte, si $L$ compte les lignes et $T$ les titres, nous vérifions :

$$
L_{\mathrm{brut}} = L_{\mathrm{corrigé}} = 2349,
\qquad T_{\mathrm{corrigé}} = 48.
$$

Attention : nous ne demandons pas la disparition de toute lettre « Â ». Le corpus contient réellement la phrase « Âme où rien n’est impur ». Un bon contrôle distingue un caractère français légitime d’un motif corrompu comme `Â«`.


In [6]:
motifs_mojibake_residuels = ("Ã", "â€", "Å“", "Â«", "Â»", "\ufffd")

assert len(positions_a_reparees) == 255
assert len(erreurs_titres) == 0
assert len(texte_corrige.splitlines()) == len(texte_brut.splitlines()) == 2349
assert len(titres_corriges) == len(titres_bruts) == 48
assert len(set(titres_corriges)) == 48

for motif in motifs_mojibake_residuels:
    assert motif not in texte_corrige, f"Motif résiduel : {motif!r}"
    assert motif not in texte_titres_corrige, f"Motif résiduel dans les titres : {motif!r}"

# Le seul « Â » restant est une vraie majuscule accentuée.
assert texte_corrige.count("Â") == 1
assert "Âme où rien n'est impur" in texte_corrige
assert "Â" not in texte_titres_corrige

temoins = [
    ("frÃªles", "frêles"),
    ("Ã©tincelles", "étincelles"),
    ("siÃ¨cle", "siècle"),
    ("cÅ“ur", "cœur"),
    ("Lâ€™enfant", "L’enfant"),
]
for forme_abimee, forme_corrigee in temoins:
    assert forme_abimee in (texte_brut + texte_titres_brut)
    assert forme_corrigee in (texte_corrige + texte_titres_corrige)

# Relecture des fichiers : leur contenu binaire doit être inchangé.
assert hashlib.sha256(CHEMIN_CORPUS.read_bytes()).hexdigest() == empreinte_corpus_source
assert hashlib.sha256(CHEMIN_TITRES.read_bytes()).hexdigest() == empreinte_titres_source

print("Réparation validée sans modification des fichiers sources.")
print("Lignes du corpus :", len(texte_corrige.splitlines()))
print("Titres uniques   :", len(set(titres_corriges)))
print()
print("Avant :")
print("\n".join(texte_brut.splitlines()[:4]))
print()
print("Après :")
print("\n".join(texte_corrige.splitlines()[:4]))


Réparation validée sans modification des fichiers sources.
Lignes du corpus : 2349
Titres uniques   : 48

Avant :
Mes vers fuiraient

Mes vers fuiraient, doux et frÃªles, Vers votre jardin si beau,
Si mes vers avaient des ailes,

Après :
Mes vers fuiraient

Mes vers fuiraient, doux et frêles, Vers votre jardin si beau,
Si mes vers avaient des ailes,


## 7. Reconstruire les 48 poèmes

### 7.1 Pourquoi une découpe automatique serait trompeuse

Trois séparateurs apparemment naturels ont été vérifiés puis écartés :

- les 48 titres ne figurent pas tous littéralement dans le corpus ;
- les suites de lignes vides séparent aussi des strophes ;
- les 47 mentions « Victor Hugo » ne représentent pas 47 fins fiables.

Deux anomalies sont particulièrement importantes. Une attribution coupe **Les Djinns** en plein milieu, et une autre coupe **Oceano Nox**. Inversement, aucune attribution ne sépare **La chasse du Burgave** de **Jeanne était au pain sec**.

Pour ce fichier fixe, nous utilisons donc un **manifeste relu** de 48 intervalles. Une borne $(d_i,f_i)$ est semi-ouverte : elle conserve les indices de ligne $j$ tels que

$$
d_i \leq j < f_i.
$$

Ce choix évite les erreurs silencieuses. Les quatre lignes bibliographiques insérées à l’intérieur de deux poèmes sont nommées et retirées explicitement. Ce découpage ne crée encore ni jetons, ni identifiants numériques, ni vocabulaire.


In [7]:
lignes_corrigees = texte_corrige.splitlines()

# Indices Python : début inclus, fin exclue.
BORNES_POEMES = [
    (0, 13),       #  1. Mes vers fuiraient
    (15, 69),      #  2. Ce siècle avait deux ans
    (74, 149),     #  3. Le mariage de Roland
    (153, 159),    #  4. L’enfant
    (168, 263),    #  5. La ronde du sabbat
    (281, 301),    #  6. Un jour je vis
    (311, 344),    #  7. Melancholia
    (351, 365),    #  8. L’hiver
    (370, 380),    #  9. J’eus toujours de l’amour
    (390, 410),    # 10. Chanson
    (417, 588),    # 11. La chasse du Burgave
    (588, 605),    # 12. Jeanne était au pain sec
    (611, 652),    # 13. Lorsque l’enfant paraît...
    (665, 680),    # 14. Ordre du jour de floréal
    (688, 726),    # 15. Spectacle rassurant
    (735, 770),    # 16. À l’obéissance passive (I)
    (778, 969),    # 17. Fonction du poète
    (983, 998),    # 18. Fable ou histoire
    (1008, 1034),  # 19. Regardez : les enfants sont assis en rond
    (1041, 1081),  # 20. J’aime l’araignée et j’aime l’ortie
    (1086, 1224),  # 21. Les Djinns
    (1224, 1245),  # 22. Printemps
    (1252, 1262),  # 23. Unité
    (1268, 1299),  # 24. Les raisons du Momotombo
    (1303, 1321),  # 25. Sur une barricade...
    (1334, 1346),  # 26. La lune
    (1350, 1373),  # 27. Clair de lune
    (1380, 1410),  # 28. Vieille chanson du jeune temps
    (1416, 1440),  # 29. La coccinelle
    (1453, 1489),  # 30. Jeanne endormie
    (1500, 1526),  # 31. La pauvre fleur disait au papillon céleste
    (1535, 1731),  # 32. Les pauvres gens
    (1746, 1760),  # 33. À une femme
    (1766, 1828),  # 34. Guitare
    (1842, 1850),  # 35. Mes deux filles
    (1855, 1867),  # 36. Viens ! – une flûte invisible
    (1873, 1900),  # 37. Chanson de grand-père
    (1904, 1917),  # 38. La vache
    (1931, 1947),  # 39. Demain, dès l’aube
    (1950, 1987),  # 40. Je suis des bois l’hôte fidèle
    (1994, 2111),  # 41. Tristesse d’Olympio
    (2119, 2185),  # 42. Ce que dit le public
    (2191, 2209),  # 43. La source tombait du rocher
    (2221, 2234),  # 44. Le poème du jardin des plantes
    (2240, 2259),  # 45. Les enfants gâtés
    (2272, 2283),  # 46. Après la bataille
    (2288, 2331),  # 47. Oceano Nox
    (2331, 2346),  # 48. Ceux qui vivent, ce sont ceux qui luttent
]

# Métadonnées placées à tort au milieu d’un poème (indices 0-based).
METADONNEES_INTERNES = {
    1153: "Victor Hugo:",
    1158: "Les Orientales",
    1159: "1829",
    2304: "Victor Hugo: Les Rayons et les ombres juillet 1836",
}

assert len(BORNES_POEMES) == len(titres_corriges) == 48
for indice, valeur_attendue in METADONNEES_INTERNES.items():
    assert lignes_corrigees[indice].strip() == valeur_attendue

poemes = []
indices_des_plages = set()
indices_de_contenu = set()

for numero, (titre, bornes) in enumerate(
    zip(titres_corriges, BORNES_POEMES),
    start=1,
):
    debut, fin = bornes
    assert 0 <= debut < fin <= len(lignes_corrigees)

    indices_courants = set(range(debut, fin))
    assert indices_des_plages.isdisjoint(indices_courants)
    indices_des_plages.update(indices_courants)

    indices_utiles = [
        indice
        for indice in range(debut, fin)
        if indice not in METADONNEES_INTERNES
    ]
    indices_de_contenu.update(indices_utiles)
    texte_poeme = "\n".join(
        lignes_corrigees[indice] for indice in indices_utiles
    ).strip()

    assert texte_poeme, f"Poème vide : {titre}"
    poemes.append({
        "numero": numero,
        "titre": titre,
        "texte": texte_poeme,
        "ligne_debut": debut + 1,
        "ligne_fin": fin,
    })


### 7.2 Vérifier que la reconstruction est complète et cohérente

Le manifeste n’est accepté que si les 48 plages sont ordonnées, disjointes et non vides. Nous contrôlons aussi plusieurs ancres situées aux endroits difficiles, notamment juste avant et après les séparations manquantes ou déplacées.

Enfin, chaque ligne non vide reçoit exactement l’une de ces deux étiquettes :

$$
\mathcal{L}_{\mathrm{non\ vide}}
= \mathcal{L}_{\mathrm{poèmes}}
\,\dot\cup\,
\mathcal{L}_{\mathrm{métadonnées}},
$$

où $\dot\cup$ indique une union sans chevauchement. Ici, 1 287 lignes non vides appartiennent aux poèmes et 113 sont des titres de source, attributions, collections ou dates volontairement exclues. Nous validons ainsi la structure, pas la qualité littéraire ni le futur vocabulaire.


In [8]:
assert len(poemes) == 48
assert [poeme["titre"] for poeme in poemes] == titres_corriges
assert all(poeme["texte"] for poeme in poemes)
assert all(
    BORNES_POEMES[indice][1] <= BORNES_POEMES[indice + 1][0]
    for indice in range(len(BORNES_POEMES) - 1)
)
assert all("Victor Hugo" not in poeme["texte"] for poeme in poemes)

ancres_debut = {
    1: "Mes vers fuiraient",
    5: "À M. Charles Nodier",
    12: "Jeanne était au pain sec",
    21: "Murs, ville",
    22: "C'est la jeunesse et le matin",
    47: "Oh ! combien de marins",
    48: "Ceux qui vivent, ce sont ceux qui luttent",
}
for numero, ancre in ancres_debut.items():
    assert poemes[numero - 1]["texte"].startswith(ancre)

indices_non_vides = {
    indice
    for indice, ligne in enumerate(lignes_corrigees)
    if ligne.strip()
}
indices_poemes_non_vides = indices_de_contenu & indices_non_vides
indices_metadonnees_non_vides = indices_non_vides - indices_poemes_non_vides

assert indices_poemes_non_vides.isdisjoint(indices_metadonnees_non_vides)
assert indices_poemes_non_vides | indices_metadonnees_non_vides == indices_non_vides
assert len(indices_poemes_non_vides) == 1287
assert len(indices_metadonnees_non_vides) == 113

print("Nombre exact de poèmes reconstruits :", len(poemes))
print("Lignes non vides de poésie          :", len(indices_poemes_non_vides))
print("Lignes bibliographiques exclues     :", len(indices_metadonnees_non_vides))

for numero in (1, 12, 21, 47, 48):
    poeme = poemes[numero - 1]
    premiere_ligne = poeme["texte"].splitlines()[0]
    print(f"{numero:>2}. {poeme['titre']} -> {premiere_ligne[:70]}")


Nombre exact de poèmes reconstruits : 48
Lignes non vides de poésie          : 1287
Lignes bibliographiques exclues     : 113
 1. Mes vers fuiraient -> Mes vers fuiraient
12. Jeanne était au pain sec -> Jeanne était au pain sec dans le cabinet noir,
21. Les Djinns -> Murs, ville Et port, Asile
47. Oceano Nox -> Oh ! combien de marins, combien de capitaines Qui sont partis joyeux p
48. Ceux qui vivent , ce sont ceux qui luttent -> Ceux qui vivent, ce sont ceux qui luttent ; ce sont Ceux dont un desse


## Bilan de l’étape 2

La réparation a été effectuée uniquement dans de nouvelles variables Python. Les deux fichiers sources et les variables brutes ont été conservés. Les 255 séquences endommagées correspondant à « à » ont été identifiées avant correction ; aucune autre erreur binaire n’a été tolérée. Les motifs de mojibake ont disparu, tandis que le vrai mot « Âme » a été préservé.

Le corpus a ensuite été reconstruit en **48 poèmes non vides**, associés dans l’ordre aux **48 titres uniques**. Le manifeste explicite documente les frontières et traite les deux attributions placées au milieu de « Les Djinns » et « Oceano Nox », ainsi que la séparation absente avant « Jeanne était au pain sec ».

À ce stade, la variable `poemes` contient seulement des dictionnaires avec un numéro, un titre, un texte et ses lignes d’origine. **Aucun tokenizer et aucun vocabulaire n’ont été créés.**

**Prochaine étape :** séparer les 48 poèmes en entraînement, validation et test sans fuite de données, puis seulement construire le vocabulaire à partir de l’entraînement.
